In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [3]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [19]:
workflow_deep_dive = workflow_research('2026-03-02','2026-03-03')

In [20]:
workflow_deep_dive

[{'event_id': '1898_2026-03-01',
  'output': 'Bloomberg (Mar 1, 2026) reports Apple plans to replace Core ML with a modern "Core AI" framework at WWDC 2026. The leak says Apple will unify and modernize on-device AI APIs (Core AI) — a change that could affect how apps integrate AI, enable new on-device inference and privacy controls, and influence tooling available to product, marketing, and design teams that build or rely on AI-enabled apps.',
  'sources': [{'url': 'https://www.bloomberg.com/news/newsletters/2026-03-01/apple-touch-screen-macbook-pro-fall-2026-details-cheap-macbook-launch-core-ai-mm7rcg48',
    'name': 'Bloomberg'}],
  'news_date': '2026-03-01',
  'topic': 'Workflow improvement'},
 {'event_id': '1870_2026-02-26',
  'output': 'Deloitte announced Enterprise AI Navigator, an end-to-end solution (launched 26 Feb 2026) to help organizations identify, prioritize, prototype, and measure AI/agent investments. The Navigator provides modules (AI Identifier, Impact Analyzer, Workf

In [21]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [22]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [23]:
openai_research_workflow(workflow_deep_dive[0:])

- REPORT SNAPSHOT — Publication: Bloomberg Power On newsletter by Mark Gurman, published March 1, 2026; topic includes plans to replace Core ML with a modernized Core AI framework at WWDC 2026. [Source: Bloomberg, https://www.bloomberg.com/news/newsletters/2026-03-01/apple-touch-screen-macbook-pro-fall-2026-details-cheap-macbook-launch-core-ai-mm7rcg48?srnd=undefined]

- REPORT SNAPSHOT — Scope: geography, sectors, sample size, time window, and method type not disclosed in cited sources. [Source: Bloomberg, https://www.bloomberg.com/news/newsletters/2026-03-01/apple-touch-screen-macbook-pro-fall-2026-details-cheap-macbook-launch-core-ai-mm7rcg48?srnd=undefined]

- CORE FINDINGS (MOST IMPORTANT) 1 — Core ML is slated to be replaced by a modernized Core AI framework for iOS 27, with details anticipated to surface at WWDC 2026. [Source: Bloomberg, https://www.bloomberg.com/news/newsletters/2026-03-01/apple-touch-screen-macbook-pro-fall-2026-details-cheap-macbook-launch-core-ai-mm7rcg48?sr